In [1]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')

print(train_full.shape, test.shape)

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

(749609, 34) (114111, 34)


In [2]:
# ============================================================
# 0. 라이브러리 불러오기
# ============================================================

import numpy as np

from catboost import CatBoostRegressor
from sklearn.metrics import root_mean_squared_error

In [3]:
# ============================================================
# 1. 컬럼군 정의
# ============================================================

target_col = 'target_maxrise_6h'  # 모델링 학습부터는 반드시 빼내야 함

# 수위 컬럼군
wl_cols = [
    'wl_t',
    'wl_t_minus_1h',
    'wl_t_minus_3h',
    'wl_t_minus_6h',
    'wl_t_minus_12h',
    'wl_t_minus_24h',
]

# 수위차분(wl_diff) 컬럼군
wl_diff_cols = [
    'wl_t',
    'wl_diff_1h',
    'wl_diff_3h',
    'wl_diff_6h',
    'wl_diff_12h',
    'wl_diff_24h'
]


# 강수 컬럼군
rain_cols = [
    # IMERG강우
    'imerg_rain_1h',
    'imerg_rain_6h',
    'imerg_rain_24h',
    # 지상강우
    'sfc_rain_1h',
    'sfc_rain_6h',
    'sfc_rain_24h',
    # 레이더 반사도
    'radar_reflectivity_mean',
    'radar_reflectivity_p95',
    'radar_reflectivity_ge20_fraction'
]

# 강우강도 컬럼군
rain_int_cols = [
    'imerg_rain_1h',
    'imerg_int_1to6h',
    'imerg_int_6to24h',
    'sfc_rain_1h',
    'sfc_int_1to6h',
    'sfc_int_6to24h'
]

# 환경 컬럼군
env_cols = [
    'cat_area',       # km²
    'wsarea',         # km²
    'mean_elevation', # m
    'mean_slope',     # °
    'urban_ratio',    # %
    'forest_ratio',   # %
    'stream_order'
]

num_cols = wl_cols + rain_cols + env_cols
num_cols_fe_wl = num_cols + [
    col for col in wl_diff_cols if col != 'wl_t'
]
num_cols_fe = num_cols_fe_wl + [
    col for col in rain_int_cols if col not in ['imerg_rain_1h', 'sfc_rain_1h']
]

print(f"""<변수개수 검증>
    수위변수: {len(wl_cols)}
    강우변수: {len(rain_cols)}, 
    환경변수: {len(env_cols)}, 
    수위차분(wl_t포함): {len(wl_diff_cols)},
    강우강도(rain_1h포함): {len(rain_int_cols)}
    기존 숫자열(수위 + 강우 + 환경): {len(num_cols)}, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): {len(num_cols_fe_wl)},
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): {len(num_cols_fe)}
""")

<변수개수 검증>
    수위변수: 6
    강우변수: 9, 
    환경변수: 7, 
    수위차분(wl_t포함): 6,
    강우강도(rain_1h포함): 6
    기존 숫자열(수위 + 강우 + 환경): 22, 
    수위차분추가 숫자열(수위 + 강우 + 환경 + 수위차분(wl_t제외)): 27,
    강우강도추가 숫자열(수위차분추가 숫자열 + 강우강도(rain_1h 2개제외)): 31



In [4]:
# ============================================================
# 2. 종속변수 설정
# ============================================================

y_train_full = train_full[target_col]
y_test = test[target_col]

In [12]:
# ============================================================
# 3. station_id 원핫인코딩 독립변수 생성
# ============================================================

# OHE 전 독립변수 데이터셋
X_train_full = train_full[num_cols_fe]
X_test = test[num_cols_fe]

# 원핫인코딩
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(
    handle_unknown='ignore',  # train에 없던 범주가 나와도 에러 X
    sparse_output=True  # sparse: 희소한, 희소행렬(0이 아닌 값이 드문 행렬)
    # 원핫인코딩 결과를 전체 다 저장 X, 희소한 1의 값의 위치만 저장(메모리 사용 적음)
)

# 학습은 train만
ohe.fit(train_full[['station_id']])  # sklearn 인코더는 2차원 행렬을 받는다.

# 변환
train_st_id_ohe = ohe.transform(train_full[['station_id']])
test_st_id_ohe = ohe.transform(test[['station_id']])
# 여기까지는 일반적은 인코딩 과정

# train, val 셋에 합치기
from scipy.sparse import hstack
# scipy안에 희소행렬을 다루는 서브캐키지 scipy.sparse
# hstack(horizontal stack)은 concat(axis=1)과 같은 역할, df가 아닌 행렬도 붙일 수 있는 함수

X_train_with_station_ohe = hstack([
    X_train_full,
    train_st_id_ohe
]).tocsr()  # df + 희소행렬(ohe) -> csr형태로 저장하여 sparse 상태 유지
# 이 변수는 더이상 df가 아님

X_test_with_station_ohe = hstack([
    X_test,
    test_st_id_ohe
]).tocsr()

print("원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남")
print(X_train_full.shape, X_train_with_station_ohe.shape)
print(X_test.shape, X_test_with_station_ohe.shape)
# 제공된 데이터셋의 관측소 개수는 465개(md파일에 명시)
# 기존컬럼 31개 + 465 = 496이 정상

원핫인코딩 전 후 데이터셋 크기 비교: 행은 같고 열만 들어남
(749609, 31) (749609, 496)
(114111, 31) (114111, 496)


In [5]:
# ============================================================
# 1. Tidal / Normal 데이터 분리
# ============================================================

# 조석 영향 의심 관측소 9개 (기존과 동일)
tidal_stations = [
    'ST0110', 'ST0326', 'ST0159',
    'ST0358', 'ST0347', 'ST0415',
    'ST0292', 'ST0313', 'ST0234'
]

# Train 분리
tidal_train = train_full.loc[
    train_full['station_id'].isin(tidal_stations)
].copy()

normal_train = train_full.loc[
    ~train_full['station_id'].isin(tidal_stations)
].copy()

# Test 분리
tidal_test = test.loc[
    test['station_id'].isin(tidal_stations)
].copy()

normal_test = test.loc[
    ~test['station_id'].isin(tidal_stations)
].copy()

# 분리 확인
assert len(tidal_train) + len(normal_train) == len(train_full)
assert len(tidal_test) + len(normal_test) == len(test)

print(f'Train: {len(train_full):,}')
print(f'  Tidal: {len(tidal_train):,}')
print(f'  Normal: {len(normal_train):,}')

print(f'\nTest: {len(test):,}')
print(f'  Tidal: {len(tidal_test):,}')
print(f'  Normal: {len(normal_test):,}')

Train: 749,609
  Tidal: 14,251
  Normal: 735,358

Test: 114,111
  Tidal: 2,169
  Normal: 111,942


In [6]:
# ============================================================
# 2. 독립변수 / 종속변수 분리
# ============================================================

# Tidal
X_tidal_train = tidal_train[num_cols_fe + ['station_id']].copy()
X_tidal_test = tidal_test[num_cols_fe + ['station_id']].copy()

y_tidal_train = tidal_train[target_col]
y_tidal_test = tidal_test[target_col]

# Normal
X_normal_train = normal_train[num_cols_fe + ['station_id']].copy()
X_normal_test = normal_test[num_cols_fe + ['station_id']].copy()

y_normal_train = normal_train[target_col]
y_normal_test = normal_test[target_col]

# CatBoost 범주형 변수 처리
for X in [
    X_tidal_train, X_tidal_test,
    X_normal_train, X_normal_test
]:
    X['station_id'] = X['station_id'].astype(str)

In [9]:
# ============================================================
# 4. 모델15 (최종검증: 기존 catboost모델, y_test)
# ============================================================
# train_test 분포비교에서 이상치 제거가 한번 있었음, 이것 반영해서 기본모델 한번 돌려보기

# CatBoost 전용 데이터셋
X_train_with_station = train_full[num_cols_fe + ['station_id']].copy()
X_test_with_station = test[num_cols_fe + ['station_id']].copy()

X_train_with_station['station_id'] = X_train_with_station['station_id'].astype(str)
X_test_with_station['station_id'] = X_test_with_station['station_id'].astype(str)

# 모델 정의
model_base_no_ohe = CatBoostRegressor(
    random_state=42,
    iterations=3000,
    cat_features=['station_id']
)

# 학습
model_base_no_ohe.fit(
    X_train_with_station,
    y_train_full,
    early_stopping_rounds=100,
    eval_set=(X_test_with_station, y_test),
    verbose=200
)

# 예측
pred_test_no_ohe = model_base_no_ohe.predict(X_test_with_station)
pred_train_no_ohe = model_base_no_ohe.predict(X_train_with_station)

# 평가
rmse_test_no_ohe = root_mean_squared_error(y_test, pred_test_no_ohe)
rmse_train_no_ohe = root_mean_squared_error(y_train_full, pred_train_no_ohe)

print('Best iteration:', model_base_no_ohe.best_iteration_)
print('Tree count:', model_base_no_ohe.tree_count_)
print('Test RMSE:', rmse_test_no_ohe)
print('Train RMSE:', rmse_train_no_ohe)
print('Gap:', rmse_test_no_ohe - rmse_train_no_ohe)

Learning rate set to 0.073655
0:	learn: 0.3815887	test: 0.3705444	best: 0.3705444 (0)	total: 107ms	remaining: 5m 22s
200:	learn: 0.2245861	test: 0.2176650	best: 0.2176650 (200)	total: 21.2s	remaining: 4m 55s
400:	learn: 0.2136080	test: 0.2114690	best: 0.2114690 (400)	total: 42.7s	remaining: 4m 36s
600:	learn: 0.2069185	test: 0.2087125	best: 0.2087125 (600)	total: 1m 7s	remaining: 4m 30s
800:	learn: 0.2019976	test: 0.2074117	best: 0.2073848 (791)	total: 1m 31s	remaining: 4m 12s
1000:	learn: 0.1980992	test: 0.2066846	best: 0.2066724 (984)	total: 1m 55s	remaining: 3m 50s
1200:	learn: 0.1949022	test: 0.2060690	best: 0.2060264 (1190)	total: 2m 18s	remaining: 3m 27s
1400:	learn: 0.1921277	test: 0.2058874	best: 0.2058632 (1392)	total: 2m 42s	remaining: 3m 5s
1600:	learn: 0.1894397	test: 0.2055767	best: 0.2055767 (1600)	total: 3m 5s	remaining: 2m 41s
1800:	learn: 0.1871858	test: 0.2052672	best: 0.2052343 (1749)	total: 3m 25s	remaining: 2m 16s
2000:	learn: 0.1850077	test: 0.2050841	best: 0.2050

In [ ]:
# ============================================================
# 3. Tidal / Normal 모델 개별 학습
# ============================================================

models_split = {}
pred_test_split = pd.Series(index=test.index, dtype=float)

for name, X_train, y_train, X_test, y_test_part in [
    ('Tidal', X_tidal_train, y_tidal_train,
     X_tidal_test, y_tidal_test),

    ('Normal', X_normal_train, y_normal_train,
     X_normal_test, y_normal_test)
]:

    print(f'\n========== {name} 학습 ==========')

    model = CatBoostRegressor(
        random_state=42,
        iterations=3000,
        cat_features=['station_id']
    )

    model.fit(
        X_train,
        y_train,
        early_stopping_rounds=100,
        eval_set=(X_test, y_test_part),
        verbose=200
    )

    pred_test = model.predict(X_test)
    pred_train = model.predict(X_train)

    # 전체 예측값에 저장
    pred_test_split.loc[X_test.index] = pred_test

    rmse_test = root_mean_squared_error(y_test_part, pred_test)
    rmse_train = root_mean_squared_error(y_train, pred_train)

    models_split[name] = model

    print(f'\n{name} 결과')
    print('Best iteration:', model.best_iteration_)
    print('Test RMSE:', rmse_test)
    print('Train RMSE:', rmse_train)
    print('Gap:', rmse_test - rmse_train)



========== Tidal 학습 ==========
Learning rate set to 0.039537
0:	learn: 1.7085461	test: 1.7659914	best: 1.7659914 (0)	total: 116ms	remaining: 5m 47s
200:	learn: 0.4056550	test: 0.4337543	best: 0.4337543 (200)	total: 6.22s	remaining: 1m 26s
400:	learn: 0.3459636	test: 0.3889485	best: 0.3889485 (400)	total: 12.4s	remaining: 1m 20s
600:	learn: 0.3132075	test: 0.3745648	best: 0.3745648 (600)	total: 18.5s	remaining: 1m 13s
800:	learn: 0.2895007	test: 0.3653392	best: 0.3653392 (800)	total: 24.6s	remaining: 1m 7s
1000:	learn: 0.2719728	test: 0.3620359	best: 0.3619379 (985)	total: 30.6s	remaining: 1m 1s
1200:	learn: 0.2570926	test: 0.3601577	best: 0.3601528 (1193)	total: 36.5s	remaining: 54.7s
1400:	learn: 0.2445987	test: 0.3590226	best: 0.3589787 (1387)	total: 42.8s	remaining: 48.9s
1600:	learn: 0.2342170	test: 0.3579406	best: 0.3579406 (1600)	total: 49s	remaining: 42.8s
1800:	learn: 0.2249139	test: 0.3571290	best: 0.3571289 (1799)	total: 55.1s	remaining: 36.7s
2000:	learn: 0.2165070	test: 0.

In [8]:
# ============================================================
# 4. 전체 Test RMSE 비교
# ============================================================

# 예측 누락 검증
assert pred_test_split.notna().all()

# 분리모델 전체 RMSE
rmse_test_split = root_mean_squared_error(
    test[target_col],
    pred_test_split
)

print('========== Test RMSE 비교 ==========')
# print(f'기존 단일 CatBoost : {rmse_test_no_ohe:.6f}')
print(f'Tidal/Normal 분리 : {rmse_test_split:.6f}')
# print(f'RMSE 변화         : {rmse_test_split - rmse_test_no_ohe:+.6f}')

========== Test RMSE 비교 ==========
Tidal/Normal 분리 : 0.203776


In [10]:
# ============================================================
# 단일 CatBoost vs Tidal/Normal 분리모델 비교
# ============================================================

# 단일모델 예측값을 row_id 인덱스로 정렬
pred_test_no_ohe = pd.Series(
    pred_test_no_ohe,
    index=test.index
)

# 집단별 비교
for name, mask in [  # mask는 불리언조건문을 가리키는 대명사로 쓰임
    ('Tidal', test['station_id'].isin(tidal_stations)),
    ('Normal', ~test['station_id'].isin(tidal_stations)),
    ('전체', pd.Series(True, index=test.index))
]:
    y = test.loc[mask, target_col]

    pred_base = pred_test_no_ohe.loc[mask]
    pred_split = pred_test_split.loc[mask]

    sse_base = ((y - pred_base) ** 2).sum()
    sse_split = ((y - pred_split) ** 2).sum()

    print(f'\n========== {name} ==========')
    print('단일모델 RMSE:', root_mean_squared_error(y, pred_base))
    print('분리모델 RMSE:', root_mean_squared_error(y, pred_split))
    print('단일모델 SSE:', sse_base)
    print('분리모델 SSE:', sse_split)
    print('SSE 감소량:', sse_base - sse_split)


========== Tidal ==========
단일모델 RMSE: 0.3803362228478671
분리모델 RMSE: 0.3557756979848764
단일모델 SSE: 313.7580883876857
분리모델 SSE: 274.54409724300183
SSE 감소량: 39.21399114468386

========== Normal ==========
단일모델 RMSE: 0.19987666576903726
분리모델 RMSE: 0.19969185050509464
단일모델 SSE: 4472.159190594013
분리모델 SSE: 4463.892681273523
SSE 감소량: 8.266509320490513

========== 전체 ==========
단일모델 RMSE: 0.20479475904370895
분리모델 RMSE: 0.2037763549933682
단일모델 SSE: 4785.9172789816985
분리모델 SSE: 4738.436778516523
SSE 감소량: 47.48050046517528


In [ ]:
# 모델 구조 선정: 단일 CatBoost와 Tidal/Normal 분리모델의 K-Fold OOF RMSE를 비교한다. 
# Test 결과도 참고하되, K-Fold에서 더 안정적이고 우수한 구조를 우선한다.


# Early Stopping 적용: K-Fold 또는 기존 단일 홀드아웃 Validation에서 확인한 best_iteration_을 활용해 최종 반복 횟수를 결정한다.
# Tidal과 Normal 각각의 최적 반복 횟수를 별도로 확정